# XGBoost Symptom Classifier — Test Run & Evaluation

**Purpose:** Load the trained XGBoost symptom classifier (from notebook 02) and evaluate it on handcrafted symptom sets for each mango disease.

This notebook:
1. Loads a saved XGBoost model (`symptom_classifier_leaf.json` or `symptom_classifier_fruit.json`)
2. Tests predictions on known symptom sets for each disease class
3. Visualizes the probability distributions per test case
4. Tests the ensemble fusion logic (image CNN score + symptom score)
5. Checks feature importances from the trained model
6. Runs edge-case tests (empty symptoms, unknown symptoms, etc.)

> **Prerequisite:** Run notebook `02_train_symptom_classifier.ipynb` first to generate the model files.

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Imports OK')

## Configuration

In [ ]:
# Change to 'fruit' to test the fruit model instead
DISEASE_TYPE = 'leaf'

MODEL_FILE = f'symptom_classifier_{DISEASE_TYPE}.json'
META_FILE  = f'symptom_classifier_{DISEASE_TYPE}_meta.json'

# ── Vocabulary ──────────────────────────────────────────────────────────────
LEAF_SYMPTOMS = [
    'dark_spots_brown',
    'dark_spots_with_yellow_halo',
    'concentric_rings_on_lesion',
    'black_specks_in_lesion',
    'pink_spore_masses',
    'lesions_enlarge_rapidly',
    'spots_with_irregular_border',
    'twig_dieback_from_tip',
    'canker_lesions_on_twig',
    'bark_cracking',
    'wilting_shoot_tips',
    'sparse_foliage',
    'white_powder_coating',
    'leaf_distortion',
    'premature_leaf_drop',
    'black_sooty_coating',
    'sooty_deposit_wiped_off',
    'yellow_discoloration',
    'brown_leaf_margins',
    'brown_leaf_tips',
    'water_soaked_lesions',
    'leaf_curling',
]

FRUIT_SYMPTOMS = [
    'black_sunken_lesions',
    'brown_patches_spreading',
    'surface_cracks_radiating',
    'pink_spore_masses_on_lesion',
    'lesion_ring_pattern',
    'soft_rot_spreading',
    'stem_end_rot',
    'shriveling_of_fruit',
    'premature_fruit_drop',
    'white_powder_on_fruit',
    'black_sooty_deposit_on_fruit',
    'fruit_discoloration',
]

LEAF_DISEASES  = ['Anthracnose', 'Die Back', 'Healthy', 'Powdery Mildew', 'Sooty Mold']
FRUIT_DISEASES = ['Anthracnose', 'Healthy']

print(f'Disease type : {DISEASE_TYPE}')
print(f'Model file   : {MODEL_FILE}')
print(f'Meta file    : {META_FILE}')

## Step 1: Load Model & Metadata

In [ ]:
try:
    model = XGBClassifier()
    model.load_model(MODEL_FILE)
    print(f'Model loaded from: {MODEL_FILE}')
except FileNotFoundError:
    raise FileNotFoundError(
        f'Model file "{MODEL_FILE}" not found.\n'
        f'Please run notebook 02_train_symptom_classifier.ipynb first to train and save the model.'
    )

try:
    with open(META_FILE, 'r') as f:
        meta = json.load(f)
    print(f'Meta loaded from : {META_FILE}')
except FileNotFoundError:
    raise FileNotFoundError(
        f'Metadata file "{META_FILE}" not found.\n'
        f'Please run notebook 02_train_symptom_classifier.ipynb first.'
    )

print()
print(f"Test accuracy    : {meta.get('test_accuracy', 'N/A')}")
print(f"Training samples : {meta.get('n_samples', 'N/A')}")
print(f"Training date    : {meta.get('training_date', 'N/A')}")
print(f"Classes          : {meta.get('classes', 'N/A')}")
print()
print('Model loaded successfully ✓')

## Step 2: Encoder Setup

In [ ]:
class SymptomFeatureEncoder:
    """Standalone encoder — no project imports needed.
    
    Converts a list of symptom strings into a binary feature vector
    aligned with the model's training vocabulary.
    """
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self._idx = {s: i for i, s in enumerate(vocabulary)}

    def encode(self, symptoms):
        """Return a float32 binary vector of length len(vocabulary)."""
        vec = np.zeros(len(self.vocabulary), dtype=np.float32)
        for s in symptoms:
            s = s.lower().strip().replace(' ', '_').replace('-', '_')
            if s in self._idx:
                vec[self._idx[s]] = 1.0
        return vec


# Instantiate with vocabulary matching DISEASE_TYPE
if DISEASE_TYPE == 'leaf':
    encoder = SymptomFeatureEncoder(LEAF_SYMPTOMS)
    DISEASE_NAMES = LEAF_DISEASES
else:
    encoder = SymptomFeatureEncoder(FRUIT_SYMPTOMS)
    DISEASE_NAMES = FRUIT_DISEASES

# Use class order from meta if available (safer for label alignment)
if 'classes' in meta:
    DISEASE_NAMES = meta['classes']

print(f'Encoder vocabulary size : {len(encoder.vocabulary)}')
print(f'Disease classes         : {DISEASE_NAMES}')

# Quick sanity-check: encode a known symptom
sample = encoder.encode(['dark_spots_brown', 'white_powder_coating'])
print(f'Sanity-check vector sum : {int(sample.sum())} (expected 2 if both in vocabulary)')

## Step 3: Test Cases — Inference on Known Symptom Sets

In [ ]:
LEAF_TEST_CASES = [
    {
        'label': 'Anthracnose',
        'description': 'Anthracnose (early)',
        'symptoms': [
            'dark_spots_brown',
            'dark_spots_with_yellow_halo',
            'concentric_rings_on_lesion',
            'lesions_enlarge_rapidly',
        ],
    },
    {
        'label': 'Anthracnose',
        'description': 'Anthracnose (severe)',
        'symptoms': [
            'dark_spots_brown',
            'black_specks_in_lesion',
            'pink_spore_masses',
            'lesions_enlarge_rapidly',
            'spots_with_irregular_border',
            'premature_leaf_drop',
        ],
    },
    {
        'label': 'Die Back',
        'description': 'Die Back (typical)',
        'symptoms': [
            'twig_dieback_from_tip',
            'wilting_shoot_tips',
            'bark_cracking',
            'brown_leaf_tips',
        ],
    },
    {
        'label': 'Die Back',
        'description': 'Die Back (severe)',
        'symptoms': [
            'twig_dieback_from_tip',
            'canker_lesions_on_twig',
            'bark_cracking',
            'wilting_shoot_tips',
            'sparse_foliage',
            'premature_leaf_drop',
        ],
    },
    {
        'label': 'Healthy',
        'description': 'Healthy (minimal noise)',
        'symptoms': ['yellow_discoloration'],
    },
    {
        'label': 'Healthy',
        'description': 'Healthy (no symptoms)',
        'symptoms': [],
    },
    {
        'label': 'Powdery Mildew',
        'description': 'Powdery Mildew (early)',
        'symptoms': [
            'white_powder_coating',
            'leaf_distortion',
            'leaf_curling',
        ],
    },
    {
        'label': 'Powdery Mildew',
        'description': 'Powdery Mildew (clear)',
        'symptoms': [
            'white_powder_coating',
            'leaf_distortion',
            'premature_leaf_drop',
            'yellow_discoloration',
            'leaf_curling',
        ],
    },
    {
        'label': 'Sooty Mold',
        'description': 'Sooty Mold (minimal)',
        'symptoms': [
            'black_sooty_coating',
            'sooty_deposit_wiped_off',
        ],
    },
    {
        'label': 'Sooty Mold',
        'description': 'Sooty Mold (clear)',
        'symptoms': [
            'black_sooty_coating',
            'sooty_deposit_wiped_off',
            'yellow_discoloration',
        ],
    },
]

FRUIT_TEST_CASES = [
    {
        'label': 'Anthracnose',
        'description': 'Anthracnose (early)',
        'symptoms': [
            'black_sunken_lesions',
            'brown_patches_spreading',
            'lesion_ring_pattern',
        ],
    },
    {
        'label': 'Anthracnose',
        'description': 'Anthracnose (severe)',
        'symptoms': [
            'black_sunken_lesions',
            'brown_patches_spreading',
            'surface_cracks_radiating',
            'pink_spore_masses_on_lesion',
            'lesion_ring_pattern',
            'fruit_discoloration',
        ],
    },
    {
        'label': 'Healthy',
        'description': 'Healthy (minor discoloration)',
        'symptoms': ['fruit_discoloration'],
    },
    {
        'label': 'Healthy',
        'description': 'Healthy (no symptoms)',
        'symptoms': [],
    },
]

TEST_CASES = LEAF_TEST_CASES if DISEASE_TYPE == 'leaf' else FRUIT_TEST_CASES
print(f'Loaded {len(TEST_CASES)} test cases for DISEASE_TYPE="{DISEASE_TYPE}"')

In [ ]:
results = []

for tc in TEST_CASES:
    vec = encoder.encode(tc['symptoms'])
    proba = model.predict_proba(vec.reshape(1, -1))[0]   # shape: (n_classes,)
    top_idx = int(np.argmax(proba))
    top_disease = DISEASE_NAMES[top_idx]
    top_conf = float(proba[top_idx]) * 100.0
    correct = top_disease == tc['label']
    prob_dict = {DISEASE_NAMES[i]: round(float(proba[i]) * 100, 2) for i in range(len(DISEASE_NAMES))}
    results.append({
        'description': tc['description'],
        'expected': tc['label'],
        'predicted': top_disease,
        'confidence': round(top_conf, 2),
        'correct': correct,
        'proba': prob_dict,
        'symptoms': tc['symptoms'],
    })

# Print results table
col_w = [30, 18, 18, 12, 8]
header = (
    f"{'Description':<{col_w[0]}}"
    f"{'Expected':<{col_w[1]}}"
    f"{'Predicted':<{col_w[2]}}"
    f"{'Confidence':>{col_w[3]}}"
    f"{'OK?':>{col_w[4]}}"
)
sep = '-' * sum(col_w)
print(sep)
print(header)
print(sep)
for r in results:
    mark = '✓' if r['correct'] else '✗'
    print(
        f"{r['description']:<{col_w[0]}}"
        f"{r['expected']:<{col_w[1]}}"
        f"{r['predicted']:<{col_w[2]}}"
        f"{r['confidence']:>{col_w[3]-1}.1f}%"
        f"{mark:>{col_w[4]}}"
    )
print(sep)

In [ ]:
total = len(results)
correct = sum(1 for r in results if r['correct'])
accuracy = correct / total * 100 if total > 0 else 0.0

print(f'Test cases  : {total}')
print(f'Correct     : {correct}')
print(f'Incorrect   : {total - correct}')
print(f'Accuracy    : {accuracy:.1f}%')

if accuracy == 100.0:
    print('\nAll handcrafted test cases passed ✓')
elif accuracy >= 80.0:
    print('\nGood performance — review missed cases above')
else:
    print('\nLow accuracy — model may need retraining or vocabulary revision')

## Step 4: Probability Distribution Visualization

In [ ]:
# Pick representative test cases for the grid
if DISEASE_TYPE == 'leaf':
    # One per disease class (use the first occurrence of each)
    seen = set()
    representative = []
    for r in results:
        if r['expected'] not in seen and len(representative) < 4:
            representative.append(r)
            seen.add(r['expected'])
    # Pad to 4 if fewer unique diseases covered
    while len(representative) < 4:
        representative.append(results[len(representative) % len(results)])
else:
    # Fruit: 2 diseases, show all 4 cases
    representative = results[:4] if len(results) >= 4 else results + results
    representative = representative[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, r in zip(axes, representative):
    diseases = list(r['proba'].keys())
    probs    = list(r['proba'].values())
    colors   = ['#2ecc71' if d == r['expected'] else '#e74c3c' for d in diseases]

    bars = ax.barh(diseases, probs, color=colors, edgecolor='white', height=0.6)
    ax.set_xlim(0, 110)
    ax.set_xlabel('Probability (%)', fontsize=9)
    ax.set_title(
        f"{r['description']}\nExpected: {r['expected']} | Predicted: {r['predicted']} ({r['confidence']:.1f}%)",
        fontsize=9,
    )
    for bar, prob in zip(bars, probs):
        ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height() / 2,
                f'{prob:.1f}%', va='center', fontsize=8)

    legend_handles = [
        mpatches.Patch(color='#2ecc71', label='Expected class'),
        mpatches.Patch(color='#e74c3c', label='Other class'),
    ]
    ax.legend(handles=legend_handles, fontsize=7, loc='lower right')

plt.suptitle(
    f'Symptom Classifier — Probability Distributions ({DISEASE_TYPE.capitalize()})',
    fontsize=12, fontweight='bold', y=1.01,
)
plt.tight_layout()

out_path = f'test_predictions_{DISEASE_TYPE}.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

## Step 5: Ensemble Fusion Test

In [ ]:
def fuse_image_and_symptom(image_probs, symptom_probs, disease_names):
    """Fuse CNN image probabilities with XGBoost symptom probabilities.

    Parameters
    ----------
    image_probs   : dict  {disease: percent 0-100}
    symptom_probs : dict  {disease: percent 0-100}
    disease_names : list  ordered list of disease class names

    Returns
    -------
    dict with keys: probabilities, top_prediction, confidence, reasoning
    """
    img_arr = np.array([image_probs.get(d, 0.0) for d in disease_names], dtype=np.float32)
    sym_arr = np.array([symptom_probs.get(d, 0.0) for d in disease_names], dtype=np.float32)

    top_img = disease_names[int(np.argmax(img_arr))]
    top_sym = disease_names[int(np.argmax(sym_arr))]
    img_conf = float(np.max(img_arr))
    sym_conf = float(np.max(sym_arr))

    if top_img == top_sym:
        fused = img_arr * 0.7 + sym_arr * 0.3
        reasoning = f'Image and symptoms agree on {top_img}'
    elif img_conf > 75.0:
        fused = img_arr
        reasoning = (
            f'Image highly confident ({img_conf:.1f}%), '
            f'overriding symptom suggestion ({top_sym})'
        )
    elif sym_conf > 75.0:
        fused = sym_arr
        reasoning = (
            f'Symptoms highly confident ({sym_conf:.1f}%), '
            f'overriding image suggestion ({top_img})'
        )
    else:
        fused = img_arr * 0.6 + sym_arr * 0.4
        reasoning = 'Both uncertain — expert review recommended'

    total = float(np.sum(fused))
    if total > 0:
        fused = fused / total * 100.0

    probs   = {disease_names[i]: round(float(fused[i]), 2) for i in range(len(disease_names))}
    top_idx = int(np.argmax(fused))
    return {
        'probabilities': probs,
        'top_prediction': disease_names[top_idx],
        'confidence': round(float(fused[top_idx]), 2),
        'reasoning': reasoning,
    }


# Use leaf disease names for scenarios (works for both types; adjust as needed)
SCENARIO_DISEASES = DISEASE_NAMES

# ── Scenario A: both agree on Anthracnose ──────────────────────────────────
img_A = {'Anthracnose': 85.0, 'Die Back': 8.0, 'Healthy': 4.0, 'Powdery Mildew': 2.0, 'Sooty Mold': 1.0}
sym_A = {'Anthracnose': 80.0, 'Die Back': 10.0, 'Healthy': 5.0, 'Powdery Mildew': 3.0, 'Sooty Mold': 2.0}

# ── Scenario B: image says Anthracnose, symptoms strongly say Die Back ─────
img_B = {'Anthracnose': 80.0, 'Die Back': 12.0, 'Healthy': 5.0, 'Powdery Mildew': 2.0, 'Sooty Mold': 1.0}
sym_B = {'Anthracnose': 5.0,  'Die Back': 90.0, 'Healthy': 3.0, 'Powdery Mildew': 1.0, 'Sooty Mold': 1.0}

# ── Scenario C: both uncertain ─────────────────────────────────────────────
img_C = {'Anthracnose': 55.0, 'Die Back': 30.0, 'Healthy': 8.0, 'Powdery Mildew': 5.0, 'Sooty Mold': 2.0}
sym_C = {'Anthracnose': 35.0, 'Die Back': 45.0, 'Healthy': 10.0, 'Powdery Mildew': 6.0, 'Sooty Mold': 4.0}

# For fruit model, use a 2-class subset
if DISEASE_TYPE == 'fruit':
    SCENARIO_DISEASES = DISEASE_NAMES  # ['Anthracnose', 'Healthy']
    img_A = {'Anthracnose': 85.0, 'Healthy': 15.0}
    sym_A = {'Anthracnose': 80.0, 'Healthy': 20.0}
    img_B = {'Anthracnose': 80.0, 'Healthy': 20.0}
    sym_B = {'Anthracnose': 10.0, 'Healthy': 90.0}
    img_C = {'Anthracnose': 55.0, 'Healthy': 45.0}
    sym_C = {'Anthracnose': 45.0, 'Healthy': 55.0}

scenarios = [
    ('A — Both agree on same disease', img_A, sym_A),
    ('B — Image says Anthracnose, symptoms strongly say Die Back / Healthy', img_B, sym_B),
    ('C — Both uncertain, no dominant signal', img_C, sym_C),
]

for label, img_p, sym_p in scenarios:
    result = fuse_image_and_symptom(img_p, sym_p, SCENARIO_DISEASES)
    print('=' * 60)
    print(f'Scenario {label}')
    print(f'  Image  probs : {img_p}')
    print(f'  Symptom probs: {sym_p}')
    print(f'  Reasoning    : {result["reasoning"]}')
    print(f'  Fused result : {result["probabilities"]}')
    print(f'  Top predict  : {result["top_prediction"]} ({result["confidence"]:.2f}%)')
print('=' * 60)

## Step 6: Feature Importances from Model

In [ ]:
importances = model.feature_importances_   # ndarray, one value per feature
vocabulary  = encoder.vocabulary

feat_df = pd.DataFrame({
    'symptom': vocabulary,
    'importance': importances,
}).sort_values('importance', ascending=False).reset_index(drop=True)

top_n = min(15, len(feat_df))
top_df = feat_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.85, top_n))[::-1]
bars = ax.barh(top_df['symptom'][::-1], top_df['importance'][::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Feature Importance (gain)', fontsize=10)
ax.set_title(
    f'Top {top_n} Symptom Features — XGBoost Importance ({DISEASE_TYPE.capitalize()})',
    fontsize=12, fontweight='bold',
)

for bar, val in zip(bars, top_df['importance'][::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
out_path = f'feature_importances_test_{DISEASE_TYPE}.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

# Compare with meta top_features if available
if 'top_features' in meta:
    print('\nTop features from meta (notebook 02):')
    for rank, feat in enumerate(meta['top_features'][:top_n], 1):
        print(f'  {rank:2d}. {feat}')
    print('\nTop features from loaded model (this notebook):')
    for rank, row in enumerate(feat_df.head(top_n).itertuples(), 1):
        print(f'  {rank:2d}. {row.symptom}  ({row.importance:.4f})')
else:
    print('\nNo top_features key in meta — displaying from model only:')
    print(feat_df.head(top_n).to_string(index=False))

## Step 7: Edge Cases

In [ ]:
def predict_symptoms(symptoms, label=''):
    """Helper: encode symptoms, run model, return formatted string."""
    vec   = encoder.encode(symptoms)
    proba = model.predict_proba(vec.reshape(1, -1))[0]
    top_i = int(np.argmax(proba))
    prob_str = ', '.join(f'{DISEASE_NAMES[i]}: {proba[i]*100:.1f}%' for i in range(len(DISEASE_NAMES)))
    print(f'  [{label}]')
    print(f'    Input     : {symptoms if symptoms else "(empty)"}')
    print(f'    Feature vec nonzero: {int(vec.sum())}')
    print(f'    Prediction: {DISEASE_NAMES[top_i]} ({proba[top_i]*100:.1f}%)')
    print(f'    All probs : {prob_str}')
    return proba


print('Edge Case 1: Empty symptom list')
print('  (Model should still produce a valid probability vector, no crash)')
predict_symptoms([], label='Empty list')

print()
print('Edge Case 2: Unknown / gibberish symptoms')
print('  (Unknown tokens should be silently ignored by encoder)')
predict_symptoms(
    ['xyzzy_unknown', 'not_a_real_symptom', 'FOOBAR', '   ', '---'],
    label='Gibberish only',
)

print()
print('Edge Case 3: All symptoms active')
all_syms = encoder.vocabulary
predict_symptoms(all_syms, label='All symptoms ON')

print()
print('Edge Case 4: Only ambiguous / cross-disease symptoms')
ambiguous = ['yellow_discoloration', 'brown_leaf_margins', 'premature_leaf_drop']
if DISEASE_TYPE == 'fruit':
    ambiguous = ['fruit_discoloration', 'soft_rot_spreading']
predict_symptoms(ambiguous, label='Cross-disease symptoms')

print()
print('All edge-case tests completed without errors ✓')

## Summary & Next Steps

### What was tested
- [ ] Model and metadata loaded from `.json` files produced by notebook 02
- [ ] Inference on handcrafted symptom sets for each disease class
- [ ] Probability distribution visualized and saved as `test_predictions_{type}.png`
- [ ] Ensemble fusion logic verified across agree / disagree / uncertain scenarios
- [ ] Feature importances extracted and saved as `feature_importances_test_{type}.png`
- [ ] Edge cases: empty list, gibberish, all-on, ambiguous symptoms — no crashes

### Deploying to Django (MangoSense backend)

1. **Copy model files** into the Django project:
   ```
   cp symptom_classifier_leaf.json  manggo-backend/mangosense/ML/
   cp symptom_classifier_leaf_meta.json  manggo-backend/mangosense/ML/
   cp symptom_classifier_fruit.json manggo-backend/mangosense/ML/
   cp symptom_classifier_fruit_meta.json manggo-backend/mangosense/ML/
   ```

2. **Load in Django view** (`mangosense/ML/symptom_classifier.py`):
   ```python
   from xgboost import XGBClassifier
   model = XGBClassifier()
   model.load_model('symptom_classifier_leaf.json')
   ```

3. **API endpoints** that will use this model:
   - `POST /api/predict/` — main prediction endpoint (pass `selected_symptoms` alongside the image)
   - `POST /api/save-confirmation/` — user feedback loop
   - `GET  /api/disease-statistics/` — admin dashboard aggregation

4. **Ensemble fusion**: call `fuse_image_and_symptom()` inside `ml_views.py` after both the CNN
   gate + disease model and the XGBoost classifier have returned probabilities.

5. **Retraining**: when enough verified data accumulates in `PredictionLog`, rerun notebook 02
   and replace the `.json` files — no code changes required in Django.